# Rule: **retrieve_enspreso_biomass**

**Description**

Uses the JRC ENSPRESO database to retrieve the biomass potentials with NUTS2 regions. This is later used to spatially disaggregate biomass potentials to PyPSA-Spain regions based on overlaps with NUTS2 regions (proportional to area)  

The list of available biomass is:  
- Agricultural waste  
- Manure solid, liquid  
- Residues from landscape care  
- Bioethanol barley, wheat, grain maize, oats, other cereals and rye  
- Sugar from sugar beet
- Miscanthus, switchgrass, RCG
- Willow
- Poplar
- Sunflower, soya seed
- Rape seed
- Fuelwood residues
- FuelwoodRW
- C&P_RW
- Secondary Forestry residues - woodchips
- Sawdust
- Municipal waste
- Sludge

**Outputs**

- data/enspreso_biomass/archive/2019-06-20/`ENSPRESO_BIOMASS.xlsx`

In [ ]:
######################################## Parameters

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = 'ES'

In [ ]:
##### Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os 
import sys
from matplotlib.colors import Normalize
from pathlib import Path

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### NUTS files (used here to display results at NUTS level)
gdf_NUTS2, gdf_NUTS3 = xp.load_nuts(params, spatial_domain=spatial_domain)

##### Set options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

#### **ENSPRESO scenarios**

There are three scenarios considered:  
- **ENS_Low**: biomass mobilisation is low. It assumes high barriers to biomass mobilisation, strict sustainability constraints, high costs, limited social acceptance and logistical limitations. The extraction of agricultural and forestry residues is restricted, options with higher land‑use intensity are excluded, and conservation objectives are prioritised. It represents a minimum “realistic” level of biomass availability that could be ensured without significant structural changes in agricultural and forest systems. This scenario is often used in energy system models as a lower bound to avoid overestimating the role of biomass.  
- **ENS_Med**: medium or reference scenario. It assumes a moderate level of mobilisation consistent with current or plausibly mid‑term policies, reasonable improvements in logistics and management, and standard application of sustainability criteria. It includes a higher utilisation of residues (agricultural, forestry and waste‑based) than ENS_Low, but without putting excessive pressure on productive systems. It is the default option in tools that rely on ENSPRESO data.  
- **ENS_High**: this is the most ambitious scenario. It represents a high mobilisation of the sustainable biomass potential, assuming significant improvements in agricultural and forest management, efficient logistics, high social acceptance, and an intensive use of residues and byproducts within the sustainability limits defined by the JRC. It corresponds to a demanding upper‑bound case that is typically used in sensitivity analyses of energy systems.

## `ENSPRESO_BIOMASS.xlsx`  

#### **Glossary**  
Biomass' classes with their corresponding commodity names are shown below.

In [ ]:
##### Load the file
enspreso_path = (Path(params["rootpath"]) / "data" / "enspreso_biomass" / "archive" / "2019-06-20"/ f"ENSPRESO_BIOMASS.xlsx")

# Read ENSPRESO glossary
glossary = pd.read_excel(
    enspreso_path,
    sheet_name="Glossary",
    header=1
)

# Drop the first column (no information)
glossary = glossary.iloc[:, 1:]

glossary

## **Scenario definition**  
Choose an ENSPRESO scenario and a horizon year to see the sustainable biomass available. This can be analysed independently from the output of the simulation, since it is only a visualization of the retrieved data used to build later the biomass potentials for bioenergy.

In [ ]:
# ENSPRESO scenario
ens_scenario = "ENS_Med"    # ENS_Low, ENS_Med, ENS_High

# Forestry has 6 additional scenarios, each with low medium and high categories
forest_scenario = 3    # 1-6

# Horizon year
horizon_year = 2030 # available years are: 2010, 2020, 2030, 2040, 2050

print("ENSPRESO scenario:", ens_scenario)
print("FORESTRY scenario:", forest_scenario)

#### **Agriculture land availability**  
See the total NUTS2 region's areas, land availability for bioenergy and their percentage to the total area.

In [ ]:
# AGRICULTURE
land_agr = pd.read_excel(
    enspreso_path,
    sheet_name="LAND AVAILABLE - AGR",
    header=8,
    dtype={"year": "Int64"}
)

land_agr_ES = land_agr[
    (land_agr["year"] == horizon_year) &
    (land_agr["country code"] == spatial_domain) &
    (land_agr["scenario"] == ens_scenario)
].copy()

land_agr_ES.head()

#### **Forestry land availability**  
See the total available forestry land according to the chosen category.

In [ ]:
# FORESTRY
land_for = pd.read_excel(
    enspreso_path,
    sheet_name="LAND AVAILABLE - FORESTRY",
    header=3,
)
first_col = land_for.columns[0]
land_for = land_for.rename(columns={first_col: "NUTS2"})

pretty_scenario_name = {
    "ENS_Low": "Low",
    "ENS_Med": "Medium",
    "ENS_High": "High"
}

forestry_column = f"S{forest_scenario}_{horizon_year}_{pretty_scenario_name[ens_scenario]}"

land_for_ES = land_for[land_for["NUTS2"].astype(str).str.startswith(spatial_domain)].copy()
land_for_ES.columns

if forestry_column not in land_for_ES.columns:
    raise KeyError(f"The column {forestry_column} does not exist.")

print("Available forestry land for bioenergy (1000 ha)")
land_for_ES[["NUTS2", forestry_column]].head()

#### **Spatial distribution of biomass potential for bioenergy in Spain**  
See the total land available for bioenergy in the selected scenario, disaggregated at NUTS2 level.

In [ ]:
#################### Parameters

variables_to_plot = [
    "agriculture",
    "forestry",
]

##### Plotting parameters
font_size = 15

#################### Plot

maps = []

# Merge with regions
if "agriculture" in variables_to_plot:
    agr_map = gdf_NUTS2.merge(
        land_agr_ES,
        left_on="NUTS_ID",
        right_on="NUTS 2 region code",
        how="left"
    )
    agr_column = "Total land available for bionenergy (1000 ha)"
    maps.append(("Agriculture", agr_map, agr_column))

if "forestry" in variables_to_plot:
    for_map = gdf_NUTS2.merge(
        land_for_ES[["NUTS2", forestry_column]],
        left_on="NUTS_ID",
        right_on="NUTS2",
        how="left"
    )
    maps.append(("Forestry", for_map, forestry_column))


#################### Plot

n_vars = len(maps)
n_cols = min(2, n_vars)
n_rows = int(np.ceil(n_vars / n_cols))

fig_size = [7 * n_cols, 5 * n_rows]

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=fig_size
)

axes_flat = np.atleast_1d(axes).flatten()


vmin = min(
    m[1][m[2]].min(skipna=True) for m in maps
)
vmax = max(
    m[1][m[2]].max(skipna=True) for m in maps
)


for ax, (label, gdf, column) in zip(axes_flat, maps):
    gdf.plot(
        column=column,
        cmap="YlGn",
        vmin=vmin,
        vmax=vmax,
        linewidth=0.6,
        edgecolor="black",
        legend=True,
        ax=ax,
    )

    if label == "Agriculture":
        title = (
            "Agriculture\n"
            "Land available for bioenergy (kha)\n"
            f"{ens_scenario} scenario in {horizon_year}"
        )
    else:
        title = (
            "Forestry\n"
            "Land available for bioenergy (kha)\n"
            f"S{forest_scenario} {ens_scenario} scenario in {horizon_year}"
        )

    ax.set_title(title, fontsize=font_size)
    ax.axis("off")


for ax in axes_flat[n_vars:]:
    ax.set_visible(False)


plt.tight_layout()
plt.show()

What percentage of total area is available for bioenergy from agriculture?

In [ ]:
agr_pct_column = "Bioenergy/total NUTS2 Area (%)"

agr_map[agr_pct_column] = agr_map[agr_pct_column] * 100

fig, ax = plt.subplots(
    figsize=fig_size,
    constrained_layout=True
)

agr_map.plot(
    column=agr_pct_column,
    cmap="YlGn",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    ax=ax,
)

ax.set_title(
    "Agriculture\n"
    "Share of NUTS2 area available for bioenergy (%)\n"
    f"{ens_scenario} scenario in {horizon_year}",
    fontsize=font_size
)

plt.show()

#### **Spatial distribution of a specific biomass class**
Specify a commodity from the glossary to see the potential bioenergy and the supply costs associated distributed by NUTS2 regions.

**Biomass energy potential**

In [ ]:
#################### Parameters

commodity_codes = [
    "MINBIOAGRW1",      # Agricultural waste
    "MINBIOGAS1",       # Manure solid, liquid
    #"MINBIOFRSR1a",    # Residues from landscape care
    #"MINBIOCRP11",     # Bioethanol barley, wheat, grain maize, oats, other cereals and rye 
    #"MINBIOCRP21",     # Sugar from sugar beet
    #"MINBIOCRP31",     # Miscanthus, switchgrass, RCG
    #"MINBIOCRP41",     # Willow
    #"MINBIOCRP41a",    # Poplar
    #"MINBIOLIQ1",      # Sunflower, soya seed 
    #"MINBIORPS1",      # Rape seed
    #"MINBIOFRSR1",     # Fuelwood residues
    #"MINBIOWOO",       # FuelwoodRW
    #"MINBIOWOOa",      # C&P_RW (Chips and Pellets (part of MINBIOWOO in BaU))
    #"MINBIOWOOW1",     # Secondary Forestry residues - woodchips
    #"MINBIOWOOW1a",    # Sawdust
    #"MINBIOMUN1",      # Municipal waste
    #"MINBIOSLU1",      # Sludge
]

font_size = 15


#################### Load data

ener_bio = pd.read_excel(
    enspreso_path,
    sheet_name="ENER - NUTS2 BioCom E",
    header=0,
    usecols="A:H"
)

ener_bio = ener_bio[
    (ener_bio["Year"] == horizon_year) &
    (ener_bio["Scenario"] == ens_scenario) &
    (ener_bio["NUTS0"] == spatial_domain)
].copy()


#################### Plot
PJ_to_TWh = 0.27778 
maps = []
commodity_labels = glossary.set_index("commodity")["description"].to_dict()

for code in commodity_codes:
    df = ener_bio[ener_bio["E-Comm"] == code]

    df_nuts2 = (
        df
        .groupby("NUST2", as_index=False)
        ["NUTS2 Potential available by Bio Commodity"]
        .sum()
    )

    # conversion from PJ to TWh/a
    df_nuts2["NUTS2 Potential available by Bio Commodity"] *= PJ_to_TWh

    gdf = gdf_NUTS2.merge(
        df_nuts2,
        left_on="NUTS_ID",
        right_on="NUST2",
        how="left"
    )

    maps.append((code, gdf))


# Layout: 2 columns, rows determined by the number of selected variables
n_vars = len(maps)
n_cols = min(2, n_vars)
n_rows = int(np.ceil(n_vars / n_cols))

fig_size = [6 * n_cols, 5 * n_rows]

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=fig_size,
    constrained_layout=True
)

axes_flat = np.atleast_1d(axes).flatten()


# Common color scale
all_values = [m[1]["NUTS2 Potential available by Bio Commodity"] for m in maps]
vmin = min(v.min(skipna=True) for v in all_values)
vmax = max(v.max(skipna=True) for v in all_values)
norm = Normalize(vmin=vmin, vmax=vmax)


for ax, (code, gdf) in zip(axes_flat, maps):
    gdf.plot(
        column="NUTS2 Potential available by Bio Commodity",
        cmap="YlGn",
        norm=norm,
        linewidth=0.6,
        edgecolor="black",
        legend=False,
        ax=ax,
    )

    total = gdf["NUTS2 Potential available by Bio Commodity"].sum()
    
    label = commodity_labels.get(code, code)

    ax.set_title(
        f"{label}\n(total: {total:.1f} TWh/a)",
        fontsize=font_size
    )
    ax.axis("off")


# Hide any unused subplots
for ax in axes_flat[n_vars:]:
    ax.set_visible(False)


# Shared colorbar (aligned with the visible subplots)
sm = plt.cm.ScalarMappable(cmap="YlGn", norm=norm)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes_flat[:n_vars].tolist(),
    location="right",
    shrink=0.8
)
cbar.set_label("TWh/a", fontsize=font_size)

fig.suptitle(
    f"Biomass energy potential by commodity ({ens_scenario}, {horizon_year})",
    fontsize=font_size * 1.2
)

plt.show()

**Supply costs**

In [ ]:
#################### Parameters

commodity_codes = [     ##### Description
    "MINBIOAGRW1",      # Agricultural waste
    "MINBIOGAS1",       # Manure solid, liquid
    #"MINBIOFRSR1a",    # Residues from landscape care
    #"MINBIOCRP11",     # Bioethanol barley, wheat, grain maize, oats, other cereals and rye 
    #"MINBIOCRP21",     # Sugar from sugar beet
    #"MINBIOCRP31",     # Miscanthus, switchgrass, RCG
    #"MINBIOCRP41",     # Willow
    #"MINBIOCRP41a",    # Poplar
    #"MINBIOLIQ1",      # Sunflower, soya seed 
    #"MINBIORPS1",      # Rape seed
    #"MINBIOFRSR1",     # Fuelwood residues
    #"MINBIOWOO",       # FuelwoodRW
    #"MINBIOWOOa",      # C&P_RW (Chips and Pellets (part of MINBIOWOO in BaU))
    #"MINBIOWOOW1",     # Secondary Forestry residues - woodchips
    #"MINBIOWOOW1a",    # Sawdust
    #"MINBIOMUN1",      # Municipal waste
    #"MINBIOSLU1",      # Sludge
]

#### Plotting parameters
font_size = 15


#################### Load data

cost_bio = pd.read_excel(
    enspreso_path,
    sheet_name="COST - NUTS2 BioCom",
    header=0,
    usecols="A:H"
)

cost_bio.columns = cost_bio.columns.str.strip()

cost_bio = cost_bio[
    (cost_bio["Year"] == horizon_year) &
    (cost_bio["Scenario"] == ens_scenario) &
    (cost_bio["NUTS0"] == spatial_domain)
].copy()


#################### Plot

maps = []
commodity_labels = glossary.set_index("commodity")["description"].to_dict()

GJ_to_MWh = 3.6

for code in commodity_codes:
    df = cost_bio[cost_bio["Energy commoditty"] == code]

    df_nuts2 = (
        df
        .groupby("NUTS2", as_index=False)
        .agg({
            "NUTS2 Bio Commodity Cost": "mean"
        })
    )
    # Convert from MWh/a to TWh/a
    df_nuts2["NUTS2 Bio Commodity Cost"] *= GJ_to_MWh

    gdf = gdf_NUTS2.merge(
        df_nuts2,
        left_on="NUTS_ID",
        right_on="NUTS2",
        how="left"
    )

    gdf = gdf[gdf["CNTR_CODE"] == "ES"]
    gdf = gdf[~gdf["NUTS_ID"].str.startswith("ES7")]

    maps.append((code, gdf))


# Layout: 2 columns, rows determined by the number of selected variables
n_vars = len(maps)
n_cols = min(2, n_vars)
n_rows = int(np.ceil(n_vars / n_cols))

fig_size = [6 * n_cols, 5 * n_rows]

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=fig_size,
    constrained_layout=True
)

axes_flat = np.atleast_1d(axes).flatten()


# Flatten axes for easy iteration (works for 1, 2 or more subplots)
all_values = [m[1]["NUTS2 Bio Commodity Cost"] for m in maps]
vmin = min(v.min(skipna=True) for v in all_values)
vmax = max(v.max(skipna=True) for v in all_values)

norm = Normalize(vmin=vmin, vmax=vmax)


for ax, (code, gdf) in zip(axes_flat, maps):
    gdf.plot(
        column="NUTS2 Bio Commodity Cost",
        cmap="OrRd",
        norm=norm,
        linewidth=0.6,
        edgecolor="black",
        legend=False,
        ax=ax,
    )

    mean_val = gdf["NUTS2 Bio Commodity Cost"].mean()

    
    label = commodity_labels.get(code, code)

    ax.set_title(
        f"{label}\n(mean: {mean_val:.2f} EUR/MWh)",
        fontsize=font_size
    )

    ax.axis("off")


# Hide any unused subplots
for ax in axes_flat[n_vars:]:
    ax.set_visible(False)


# Shared colorbar (aligned with the visible subplots)
sm = plt.cm.ScalarMappable(cmap="OrRd", norm=norm)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes_flat[:n_vars].tolist(),
    location="right",
    shrink=0.8
)
cbar.set_label("EUR/MWh", fontsize=font_size)

fig.suptitle(
    f"Biomass supply cost by commodity ({ens_scenario}, {horizon_year})",
    fontsize=font_size * 1.2
)

plt.show()